TensorFlow — Commands and Usage in Machine Learning

1.  Tensor Creation & Variables
2.  Tensor Attributes & Operations
3.  GradientTape — Automatic Differentiation
4.  Building Models — Sequential API
5.  Building Models — Functional API
6.  Building Models — Custom Model (subclassing)
7.  Layers & Activations
8.  Loss Functions & Metrics
9.  Optimizers
10. Training — model.fit
11. Custom Training Loop
12. tf.data — Data Pipeline

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, metrics, optimizers

print(f"TensorFlow version: {tf.__version__}")

1. Tensor Creation & Variables
tf.constant — immutable tensor (data, inputs)
tf.Variable — mutable tensor (model weights, updated during training)

In [ ]:
# tf.constant — immutable, used for data and computations
a = tf.constant([1, 2, 3])                              # infer dtype
b = tf.constant([[1.0, 2.0], [3.0, 4.0]])               # 2D float32
c = tf.constant([1, 2, 3], dtype=tf.float16)            # explicit dtype

# Filled tensors
zeros   = tf.zeros((3, 4))
ones    = tf.ones((2, 3))
full    = tf.fill((2, 2), 7.0)
eye     = tf.eye(3)

# Ranges
arange   = tf.range(0, 10, 2)
linspace = tf.linspace(0.0, 1.0, 5)

# Random
rand  = tf.random.uniform((2, 3))                        # uniform [0, 1)
randn = tf.random.normal((2, 3))                         # N(0, 1)

# From NumPy
arr      = np.array([1.0, 2.0, 3.0])
from_np  = tf.constant(arr)

print(f"constant 1D:   {a}")
print(f"zeros (3x4):\n{zeros}")
print(f"linspace:      {linspace}")
print(f"rand (2x3):\n{rand}")

# tf.Variable — mutable, used for model weights
w = tf.Variable([[0.1, 0.2], [0.3, 0.4]], dtype=tf.float32)
b_bias = tf.Variable([0.0, 0.0])

print(f"\ntf.Variable:\n{w}")

# Assign new value to variable
w.assign([[1.0, 0.0], [0.0, 1.0]])
w.assign_add([[0.1, 0.0], [0.0, 0.1]])     # w += delta
print(f"After assign_add:\n{w}")

2. Tensor Attributes & Operations
Inspect and manipulate tensors — similar to NumPy.

In [ ]:
x = tf.constant([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])

# Attributes
print(f"shape:  {x.shape}")
print(f"ndim:   {x.ndim}")
print(f"dtype:  {x.dtype}")
print(f"rank:   {tf.rank(x)}")
print(f"size:   {tf.size(x)}")

# Type casting
x_i32 = tf.cast(x, tf.int32)
x_f16 = tf.cast(x, tf.float16)
print(f"\ncast to int32:   {x_i32.dtype}")
print(f"cast to float16: {x_f16.dtype}")

# Math operations
a = tf.constant([1.0, 4.0, 9.0])
b = tf.constant([2.0, 2.0, 3.0])
print(f"\na + b:    {tf.add(a, b).numpy()}")
print(f"a * b:    {tf.multiply(a, b).numpy()}")
print(f"sqrt(a):  {tf.sqrt(a).numpy()}")
print(f"exp(b):   {tf.exp(b).numpy()}")
print(f"abs:      {tf.abs(tf.constant([-1.0, 2.0, -3.0])).numpy()}")

# Matrix operations
A = tf.constant([[1.0, 2.0], [3.0, 4.0]])
B = tf.constant([[5.0, 6.0], [7.0, 8.0]])
print(f"\nmatmul:\n{tf.matmul(A, B).numpy()}")
print(f"transpose:\n{tf.transpose(A).numpy()}")

# Reshaping
v = tf.range(12.0)
print(f"\nreshape (3,4):\n{tf.reshape(v, (3, 4)).numpy()}")

# Squeeze / expand
y = tf.constant([[1.0, 2.0, 3.0]])          # shape (1, 3)
print(f"squeeze:     {tf.squeeze(y).shape}")
print(f"expand dim0: {tf.expand_dims(tf.squeeze(y), 0).shape}")

# Reductions
print(f"\nsum:         {tf.reduce_sum(A).numpy()}")
print(f"mean axis=0: {tf.reduce_mean(A, axis=0).numpy()}")
print(f"max:         {tf.reduce_max(A).numpy()}")
print(f"argmax ax=1: {tf.argmax(A, axis=1).numpy()}")

3. GradientTape — Automatic Differentiation
TF records operations inside tf.GradientTape().
Call tape.gradient(loss, variables) to compute all derivatives.

In [ ]:
# Basic gradient
x = tf.Variable(2.0)

with tf.GradientTape() as tape:
    y = x ** 3 + 2 * x             # y = x³ + 2x

dy_dx = tape.gradient(y, x)        # dy/dx = 3x² + 2 = 14
print(f"x = {x.numpy()}")
print(f"y = x³ + 2x = {y.numpy()}")
print(f"dy/dx = {dy_dx.numpy()}  (expected: 14.0)")

# Gradient of multiple variables at once
w = tf.Variable([[0.1, 0.2], [0.3, 0.4]])
b = tf.Variable([0.0, 0.0])
X = tf.constant([[1.0, 2.0], [3.0, 4.0]])
y_true = tf.constant([[1.0, 0.0], [0.0, 1.0]])

with tf.GradientTape() as tape:
    y_pred = tf.matmul(X, w) + b
    loss   = tf.reduce_mean((y_pred - y_true) ** 2)

grads = tape.gradient(loss, [w, b])
print(f"\nLoss: {loss.numpy():.4f}")
print(f"grad w:\n{grads[0].numpy()}")
print(f"grad b: {grads[1].numpy()}")

# persistent=True — tape can be used more than once
x2 = tf.Variable(3.0)
with tf.GradientTape(persistent=True) as tape2:
    y2 = x2 ** 2
    z2 = x2 ** 3
dy2 = tape2.gradient(y2, x2)     # 2x = 6
dz2 = tape2.gradient(z2, x2)     # 3x² = 27
del tape2
print(f"\nd(x²)/dx at x=3: {dy2.numpy()}  (expected 6)")
print(f"d(x³)/dx at x=3: {dz2.numpy()}  (expected 27)")

4. Building Models — Sequential API
Fastest way to build a model: stack layers in order.
Use when each layer has exactly one input and one output.

In [ ]:
# Sequential — one-liner stack
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(8,  activation='relu'),
    layers.Dense(3),                              # raw logits, no activation
])

model.summary()

# Forward pass
x_batch = tf.random.normal((8, 4))               # 8 samples, 4 features
logits  = model(x_batch, training=False)          # training=False → no dropout
print(f"\nInput shape:  {x_batch.shape}")
print(f"Output shape: {logits.shape}")

5. Building Models — Functional API
More flexible than Sequential.
Supports multiple inputs, multiple outputs, and shared layers (e.g., ResNet skip connections).

In [ ]:
# Functional API — define the graph of layers explicitly
inputs  = keras.Input(shape=(4,), name='input')
x       = layers.Dense(16, activation='relu', name='dense_1')(inputs)
x       = layers.Dropout(0.3, name='dropout')(x)
x       = layers.Dense(8,  activation='relu', name='dense_2')(x)
outputs = layers.Dense(3,  name='logits')(x)

model_fn = keras.Model(inputs=inputs, outputs=outputs, name='mlp_functional')
model_fn.summary()

# Multiple outputs example
shared  = layers.Dense(16, activation='relu')
inp2    = keras.Input(shape=(4,))
h       = shared(inp2)
out_cls = layers.Dense(3,  name='classifier')(h)
out_reg = layers.Dense(1,  name='regressor')(h)

multi_model = keras.Model(inputs=inp2, outputs=[out_cls, out_reg])
print(f"\nMulti-output model outputs: {[o.name for o in multi_model.outputs]}")

6. Building Models — Custom Model (subclassing)
Maximum control — define layers in __init__, forward pass in call().
Mirrors the PyTorch style, useful for research and non-standard architectures.

In [ ]:
class MLP(keras.Model):
    def __init__(self, hidden, out_features):
        super().__init__()
        self.fc1  = layers.Dense(hidden, activation='relu')
        self.drop = layers.Dropout(0.3)
        self.fc2  = layers.Dense(hidden, activation='relu')
        self.fc3  = layers.Dense(out_features)

    def call(self, x, training=False):
        x = self.fc1(x)
        x = self.drop(x, training=training)   # dropout only active when training=True
        x = self.fc2(x)
        return self.fc3(x)                    # raw logits

model_custom = MLP(hidden=16, out_features=3)

# Build by running a dummy forward pass
x_dummy = tf.zeros((1, 4))
_ = model_custom(x_dummy)
model_custom.summary()

# training=True vs False
logits_train = model_custom(x_dummy, training=True)
logits_infer = model_custom(x_dummy, training=False)
print(f"\nOutput (training): {logits_train.numpy()}")
print(f"Output (inference):{logits_infer.numpy()}")

7. Layers & Activations
The building blocks of every Keras model.

In [ ]:
x = tf.random.normal((8, 16))    # batch of 8, 16 features

# --- Core Layers ---
dense  = layers.Dense(8)
print(f"Dense (16→8):     {dense(x).shape}")

# Normalization
bn = layers.BatchNormalization()
ln = layers.LayerNormalization()
print(f"BatchNorm:        {bn(x, training=True).shape}")
print(f"LayerNorm:        {ln(x).shape}")

# Regularization
drop = layers.Dropout(0.5)
print(f"Dropout:          {drop(x, training=True).shape}")

# Convolutional
x_img = tf.random.normal((4, 28, 28, 1))       # 4 images, 28x28, 1 channel
conv  = layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')
pool  = layers.MaxPooling2D(pool_size=2)
flat  = layers.Flatten()
print(f"\nConv2D:           {conv(x_img).shape}")
print(f"MaxPool2D:        {pool(conv(x_img)).shape}")
print(f"Flatten:          {flat(conv(x_img)).shape}")

# Embedding
embed     = layers.Embedding(input_dim=100, output_dim=16)
token_ids = tf.constant([[3, 7, 42], [1, 5, 9]])
print(f"\nEmbedding:        {embed(token_ids).shape}")

# Recurrent
x_seq = tf.random.normal((4, 10, 8))           # 4 samples, 10 timesteps, 8 features
lstm  = layers.LSTM(16, return_sequences=True)
gru   = layers.GRU(16)
print(f"LSTM (return_seq):{lstm(x_seq).shape}")
print(f"GRU:              {gru(x_seq).shape}")

# Activation functions as layers
v = tf.constant([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nReLU:     {tf.nn.relu(v).numpy()}")
print(f"GELU:     {[round(x, 3) for x in tf.nn.gelu(v).numpy()]}")
print(f"Sigmoid:  {[round(x, 3) for x in tf.nn.sigmoid(v).numpy()]}")
print(f"Tanh:     {[round(x, 3) for x in tf.nn.tanh(v).numpy()]}")
print(f"Softmax:  {[round(x, 3) for x in tf.nn.softmax(v).numpy()]}")

8. Loss Functions & Metrics
Losses measure error during training. Metrics measure performance for reporting.

In [ ]:
# --- Classification Losses ---

# SparseCategoricalCrossentropy: labels are integers (most common)
scce = losses.SparseCategoricalCrossentropy(from_logits=True)
logits  = tf.constant([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
targets = tf.constant([0, 1])                      # integer class labels
print(f"SparseCatCE (from logits): {scce(targets, logits).numpy():.4f}")

# CategoricalCrossentropy: labels are one-hot vectors
cce      = losses.CategoricalCrossentropy(from_logits=True)
targets_oh = tf.constant([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
print(f"CatCE (from logits):       {cce(targets_oh, logits).numpy():.4f}")

# BinaryCrossentropy: binary classification
bce      = losses.BinaryCrossentropy(from_logits=True)
logits_b = tf.constant([2.0, -1.0, 0.5])
labels_b = tf.constant([1.0,  0.0, 1.0])
print(f"BinaryCE (from logits):    {bce(labels_b, logits_b).numpy():.4f}")

# --- Regression Losses ---
preds  = tf.constant([2.5, 0.0, 2.1])
labels = tf.constant([3.0, 0.5, 2.0])

mse   = losses.MeanSquaredError()
mae   = losses.MeanAbsoluteError()
huber = losses.Huber(delta=1.0)
print(f"\nMSE:   {mse(labels, preds).numpy():.4f}")
print(f"MAE:   {mae(labels, preds).numpy():.4f}")
print(f"Huber: {huber(labels, preds).numpy():.4f}")

# --- Metrics ---
acc    = metrics.SparseCategoricalAccuracy()
auc    = metrics.AUC()

acc.update_state(targets, logits)
print(f"\nAccuracy: {acc.result().numpy():.4f}")
acc.reset_state()    # must reset between epochs

9. Optimizers
Optimizers update model weights to minimize the loss.

In [ ]:
# Common optimizers
adam    = optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999)
adamw   = optimizers.AdamW(learning_rate=1e-3, weight_decay=0.01)
sgd     = optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
rmsprop = optimizers.RMSprop(learning_rate=1e-3, rho=0.9)

print("Optimizers:")
print(f"  Adam    lr={adam.learning_rate.numpy():.4f}")
print(f"  AdamW   lr={adamw.learning_rate.numpy():.4f}")
print(f"  SGD     lr={sgd.learning_rate.numpy():.4f}")

# Learning rate schedules
lr_decay = optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3,
    decay_steps=1000,
    decay_rate=0.9
)
lr_cosine = optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=5000
)

opt_sched = optimizers.Adam(learning_rate=lr_cosine)
print(f"\nCosineDecay initial lr: {lr_cosine(0).numpy():.4f}")
print(f"CosineDecay at step 2500: {lr_cosine(2500).numpy():.6f}")

10. Training — model.fit
The high-level Keras training API.
Handles the training loop, validation, metrics, and callbacks automatically.

In [ ]:
# Synthetic dataset
np.random.seed(42)
n = 500
X_data = np.random.randn(n, 4).astype(np.float32)
y_data = (X_data[:, 0] + X_data[:, 1] > 0).astype(np.int32)  # binary labels

# Build model
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1),                   # single logit for binary classification
])

# Compile: attach loss, optimizer, metrics
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss=losses.BinaryCrossentropy(from_logits=True),
    metrics=[metrics.BinaryAccuracy(threshold=0.0)],   # threshold=0 since from_logits
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

# Train
history = model.fit(
    X_data, y_data,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nFinal train acc: {history.history['binary_accuracy'][-1]:.4f}")
print(f"Final val acc:   {history.history['val_binary_accuracy'][-1]:.4f}")

11. Custom Training Loop
Use when you need full control: custom gradient clipping,
multiple losses, complex update schedules, or research architectures.

In [ ]:
# Custom loop — mirrors the PyTorch-style but in TF

tf.random.set_seed(42)
n = 400
X_tr = tf.random.normal((n, 4))
y_tr = tf.cast(tf.random.uniform((n,), 0, 3, dtype=tf.int32), tf.int32)

model_c   = MLP(hidden=16, out_features=3)
optimizer_c = optimizers.Adam(learning_rate=1e-3)
loss_fn_c   = losses.SparseCategoricalCrossentropy(from_logits=True)
acc_metric  = metrics.SparseCategoricalAccuracy()

# @tf.function compiles the training step to a static graph → much faster
@tf.function
def train_step(X_batch, y_batch):
    with tf.GradientTape() as tape:
        logits_c = model_c(X_batch, training=True)
        loss_c   = loss_fn_c(y_batch, logits_c)
    grads = tape.gradient(loss_c, model_c.trainable_variables)
    # Gradient clipping — prevents exploding gradients (important for RNNs)
    grads, _ = tf.clip_by_global_norm(grads, clip_norm=1.0)
    optimizer_c.apply_gradients(zip(grads, model_c.trainable_variables))
    acc_metric.update_state(y_batch, logits_c)
    return loss_c

batch_size = 32
n_batches  = n // batch_size

for epoch in range(10):
    acc_metric.reset_state()
    total_loss = 0.0
    for i in range(n_batches):
        X_b = X_tr[i*batch_size : (i+1)*batch_size]
        y_b = y_tr[i*batch_size : (i+1)*batch_size]
        total_loss += train_step(X_b, y_b)

    if (epoch + 1) % 2 == 0:
        avg_loss = total_loss / n_batches
        acc_val  = acc_metric.result().numpy()
        print(f"Epoch {epoch+1:2d} | loss: {avg_loss:.4f}  acc: {acc_val:.4f}")

12. tf.data — Data Pipeline
tf.data builds fast, scalable input pipelines.
Handles batching, shuffling, prefetching, and augmentation in a graph-optimized way.

In [ ]:
# Build a dataset from tensors
X_np = np.random.randn(200, 4).astype(np.float32)
y_np = np.random.randint(0, 3, 200).astype(np.int32)

dataset = tf.data.Dataset.from_tensor_slices((X_np, y_np))

# Standard pipeline
BATCH_SIZE  = 32
BUFFER_SIZE = 200   # for shuffling
AUTOTUNE    = tf.data.AUTOTUNE

train_ds = (
    dataset
    .shuffle(BUFFER_SIZE, seed=42)   # randomize order
    .batch(BATCH_SIZE)               # group into batches
    .prefetch(AUTOTUNE)              # prepare next batch while GPU trains on current
)

# Inspect one batch
for X_batch, y_batch in train_ds.take(1):
    print(f"Batch X shape: {X_batch.shape}")
    print(f"Batch y shape: {y_batch.shape}")

# Map: apply a transformation to each element
def normalize(X, y):
    X = (X - tf.reduce_mean(X)) / tf.math.reduce_std(X)
    return X, y

train_ds_norm = (
    dataset
    .map(normalize, num_parallel_calls=AUTOTUNE)   # parallel preprocessing
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Use with model.fit
model_ds = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(3),
])
model_ds.compile(
    optimizer='adam',
    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)
history_ds = model_ds.fit(train_ds_norm, epochs=5, verbose=1)
print(f"\nFinal accuracy via tf.data pipeline: {history_ds.history['accuracy'][-1]:.4f}")